In [ ]:
import os
import pandas as pd
import time

# 라이브러리 설치: uv add pydrive2
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

# 인증 과정 (만료될 때마다 브라우저 열림)
gauth = GoogleAuth()
gauth.LocalWebserverAuth() 
drive = GoogleDrive(gauth)

def get_subfolder_list(parent_id):
    # 구글 드라이브에게 날릴 질문(Query)을 작성
    # '부모가 parent_id이고, 타입이 폴더이며, 휴지통에 있지 않은 것'
    query = f"'{parent_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = False"
    
    file_list = drive.ListFile({'q': query}).GetList()
    
    sub_folders = []
    for f in file_list:
        sub_folders.append({
            'name': f['title'],
            'id': f['id']
        })
    return sub_folders

# 각 폴더 ID를 넣으면 하위 폴더들이 list_of_folders 리스트로 들어감
list_of_folders_tongsin = get_subfolder_list('1LOTaqxasQWNucSTTPn_yGYU7A2D5cDsr')
list_of_folders_card = get_subfolder_list('14u8q6c3z2ndtWlVpI3pOYiIxRLGWHNkS')
list_of_folders_sinyong = get_subfolder_list('1rSiC1tq3dq5MGllTGlzCUhZYB_cnzVrD')
list_of_folders_company = get_subfolder_list('1FQ_6TZ2BxXoAx4Lpg9VNMzj_vlbJCHqV')

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=641874606667-v76q7nr0oa6dgm71k4q9vntrapbaeroq.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=online&response_type=code

Authentication successful.


## 기업, 신용
- utf-8-sig
- astype(str) 제거
- on_bad_lines='skip' 제거

In [ ]:
def smart_convert(csv_path):
    """인코딩을 바꿔가며 시도하고 성공 시 CSV 삭제"""
    try:
        if os.path.getsize(csv_path) == 0:
            os.remove(csv_path)
            return False
            
        parquet_path = csv_path.replace('.csv', '.parquet')
        success = False
        
        # 인코딩 utf-8-sig로 지정
        enc = 'utf-8-sig'
        try:
            # df = pd.read_csv(csv_path, encoding=enc, on_bad_lines='skip')
            df = pd.read_csv(csv_path, encoding=enc)
            # df.astype(str).to_parquet(parquet_path, engine='pyarrow', compression='snappy')
            df.to_parquet(parquet_path, engine='pyarrow', compression='snappy')
            success = True
        except:
            success = False
        
        if success:
            os.remove(csv_path) # 성공 즉시 CSV 삭제 (용량 확보)
            return True
        return False
    except:
        return False

def download_and_convert_folder(folder_id, local_path):
    if not os.path.exists(local_path):
        os.makedirs(local_path, exist_ok=True)

    query = f"'{folder_id}' in parents and mimeType != 'application/vnd.google-apps.folder' and trashed = False"
    file_list = drive.ListFile({'q': query}).GetList()

    for file_info in file_list:
        file_name = file_info['title']
        file_id = file_info['id']
        parquet_path = os.path.join(local_path, file_name.replace('.csv', '.parquet'))
        csv_save_path = os.path.join(local_path, file_name)

        # 이미 완료된 파일(parquet)이 있으면 무조건 스킵
        if os.path.exists(parquet_path):
            continue

        for attempt in range(3): # 예외가 생길 혹시 모를 경우를 대비해 3번까지 시도
            try:
                print(f"📥 다운로드 및 변환 중: {file_name}")
                file_obj = drive.CreateFile({'id': file_id})
                file_obj.GetContentFile(csv_save_path)
                
                if smart_convert(csv_save_path):
                    print(f"✅ 완료: {file_name}")
                else:
                    print(f"❌ 변환 실패: {file_name}")
                break
            except Exception as e:
                if "quotaExceeded" in str(e).lower():
                    print("구글 할당량 초과! 5분 휴식")
                    time.sleep(300)
                elif "No space left" in str(e):
                    print("하드 용량 부족! 여기서 멈춥니다.")
                    return # 용량 없으면 아예 종료
                else:
                    print(f"⚠️ 에러: {e}")
                    time.sleep(10)


### 기업

In [ ]:
# --- 메인 실행 루프 ---
base_output = "seongnam_data/기업"

for folder_info in list_of_folders_company:
    name = folder_info['name']
    f_id = folder_info['id']
    target_path = os.path.join(base_output, name)
    
    print(f"\n🚀 {name} 작업 시작...")
    download_and_convert_folder(f_id, target_path)


🚀 5. 기업 이전 통계(nps_move_cnt) 작업 시작...

🚀 4. 신규 기업(new) 작업 시작...

🚀 3. 법인 기업(cnt) 작업 시작...


### 신용

In [ ]:
# --- 메인 실행 루프 ---
base_output = "seongnam_data/신용"

for folder_info in list_of_folders_sinyong:
    name = folder_info['name']
    f_id = folder_info['id']
    target_path = os.path.join(base_output, name)
    
    print(f"\n🚀 {name} 작업 시작...")
    download_and_convert_folder(f_id, target_path)


🚀 9. 이동통계(대민)(WORK_STAT) 작업 시작...

🚀 8. 신용정보(대민)(DM_STAT) 작업 시작...

🚀 4. 전출(대민)(OUT STAT) 작업 시작...

🚀 3. 전입(대민)(IN STAT) 작업 시작...

🚀 7. 전이(경계)(CHAGE_L) 작업 시작...

🚀 6. 전이(CHANGE) 작업 시작...


## 통신 : 
- utf-8-sig 
- astype(str) 제거
- on_bad_lines='skip' 제거
- dtype 지정 (dtype=columns_types)
- engine='pyarrow' 추가
- 에러유발 가능성 큰 컬럼들 사후처리 추가

In [ ]:
columns_types = {
    # 식별자 및 코드 (문자열 또는 카테고리)
    'ETL_YM': 'str',  # 기준 년월 : 숫자로 읽으면 앞자리가 사라질 위험이 있으므로 문자열 타입이 가장 안전
    'ETL_YMD': 'str',
    'DOW': 'category', # 요일 구분 --> category
    'SEX_CD': 'category',
    'AGE_GRP': 'category',
    'PURPOSE': 'category',
    'TRANS_GB': 'category',
    
    # 명칭 (문자열)
    'O_MEGA_NM': 'str', 'D_MEGA_NM': 'str',
    'O_CTY_NM': 'str', 'D_CTY_NM': 'str',
    'O_ADMI_NM': 'str', 'D_ADMI_NM': 'str',
    
    # 코드성 숫자 (문자열 권장)
    'O_ADMI_CD': 'str', 'D_ADMI_CD': 'str', # 행정동 코드 : 숫자로 보이지만 실제로는 '코드'이므로 str 타입으로 읽는 것이 표준
    'O_CTY_CD': 'str', 'D_CTY_CD': 'str',
    
    # 시간대 (작은 정수)
    'O_TIME_CD': 'int8',
    'D_TIME_CD': 'int8',
    
    # 수치 데이터 (실수 - 결측치 대응)
    'CNT': 'float32',
    'O_CENTER_X': 'float32',
    'O_CENTER_Y': 'float32',
    'D_CENTER_X': 'float32',
    'D_CENTER_Y': 'float32',
    'DISTANCE': 'float32',
    'CARBON_EMISSIONS': 'float32'
}

def smart_convert(csv_path):
    """인코딩을 바꿔가며 시도하고 성공 시 CSV 삭제"""
    try:
        if os.path.getsize(csv_path) == 0:
            os.remove(csv_path)
            return False
            
        parquet_path = csv_path.replace('.csv', '.parquet')
        success = False

        # 인코딩 utf-8-sig로 지정
        enc = 'utf-8-sig'
        try:
            # 파일 읽기전 0행의 컬럼명만 받아오기
            actual_cols = pd.read_csv(csv_path, nrows=0).columns
            # print(actual_cols) # 0번째 행 컬럼 헤더 깨지지않고 멀쩡한지 확인

            # 실제 파일에 존재하는 컬럼만 골라내어 dtype에 전달
            current_dtypes = {k: v for k, v in columns_types.items() if k in actual_cols}
            df = pd.read_csv(csv_path, encoding=enc, dtype=current_dtypes, engine='pyarrow') # pyarrow 사용하면 속도 빨라짐

            # 에러 유발 가능성이 높은 수치형 컬럼들 사후 처리
            cols_to_fix = ['D_CENTER_Y', 'O_CENTER_Y', 'CNT', 'DISTANCE', 'CARBON_EMISSIONS']
            for col in cols_to_fix:
                if col in df.columns:
                    # 글자가 섞여있어도 NaN으로 바꾸며 float32로 변환 (메모리 절약)
                    df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

            df.to_parquet(parquet_path, engine='pyarrow', compression='snappy')
            success = True
        except Exception as e:
            print(f"❌ {csv_path} 변환 중 에러 발생: {e}") # 에러 메시지 출력!
        
        if success:
            os.remove(csv_path) # 성공 즉시 CSV 삭제 (용량 확보)
            return True
        return False
    except Exception as e:
        print(f"❌ {csv_path} 변환 중 에러 발생: {e}") # 에러 메시지 출력!
        return False

def download_and_convert_folder(folder_id, local_path):
    if not os.path.exists(local_path):
        os.makedirs(local_path, exist_ok=True)

    query = f"'{folder_id}' in parents and mimeType != 'application/vnd.google-apps.folder' and trashed = False"
    file_list = drive.ListFile({'q': query}).GetList()

    for file_info in file_list:
        file_name = file_info['title']
        file_id = file_info['id']
        parquet_path = os.path.join(local_path, file_name.replace('.csv', '.parquet'))
        csv_save_path = os.path.join(local_path, file_name)

        # 이미 완료된 파일(parquet)이 있으면 무조건 스킵
        if os.path.exists(parquet_path):
            continue

        for attempt in range(3): # 예외가 생길 혹시 모를 경우를 대비해 3번까지 시도
            try:
                print(f"📥 다운로드 및 변환 중: {file_name}")
                file_obj = drive.CreateFile({'id': file_id})
                file_obj.GetContentFile(csv_save_path)
                
                if smart_convert(csv_save_path):
                    print(f"✅ 완료: {file_name}")
                else:
                    print(f"❌ 변환 실패: {file_name}")
                break
            except Exception as e:
                if "quotaExceeded" in str(e).lower():
                    print("구글 할당량 초과! 5분 휴식")
                    time.sleep(300)
                elif "No space left" in str(e):
                    print("하드 용량 부족! 여기서 멈춥니다.")
                    return # 용량 없으면 아예 종료
                else:
                    print(f"⚠️ 에러: {e}")
                    time.sleep(10)

In [ ]:
# --- 메인 실행 루프 ---
base_output = "seongnam_data/통신"

for folder_info in list_of_folders_tongsin:
    name = folder_info['name']
    f_id = folder_info['id']
    target_path = os.path.join(base_output, name)
    
    print(f"\n🚀 {name} 작업 시작...")
    download_and_convert_folder(f_id, target_path)


🚀 T27 작업 시작...

🚀 T26 작업 시작...

🚀 T25 작업 시작...

🚀 T24 작업 시작...

🚀 T23 작업 시작...

🚀 T22 작업 시작...

🚀 T21 작업 시작...

🚀 T20 작업 시작...

🚀 T18 작업 시작...
📥 다운로드 및 변환 중: T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv
❌ seongnam_data/통신/T18/T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv 변환 중 에러 발생: CSV parse error: Expected 12 columns, got 13: 202401,월,13,41135,경기도,성남시 분당구,965120,1931210,1,W,05,6764.48,190.61
❌ 변환 실패: T18_GG_TRANS_SEXAGE_DURATION_SGG_INFLOW_202401_성남시.csv

🚀 T16 작업 시작...

🚀 T14 작업 시작...

🚀 T13 작업 시작...
📥 다운로드 및 변환 중: T13_GG_PURPOSE_SEXAGE_ADMI_OD_202512_성남시.csv
✅ 완료: T13_GG_PURPOSE_SEXAGE_ADMI_OD_202512_성남시.csv
📥 다운로드 및 변환 중: T13_GG_PURPOSE_SEXAGE_ADMI_OD_202511_성남시.csv
✅ 완료: T13_GG_PURPOSE_SEXAGE_ADMI_OD_202511_성남시.csv
📥 다운로드 및 변환 중: T13_GG_PURPOSE_SEXAGE_ADMI_OD_202510_성남시.csv
✅ 완료: T13_GG_PURPOSE_SEXAGE_ADMI_OD_202510_성남시.csv
📥 다운로드 및 변환 중: T13_GG_PURPOSE_SEXAGE_ADMI_OD_202508_성남시.csv
✅ 완료: T13_GG_PURPOSE_SEXAGE_ADMI_OD_202508_성남시.csv
📥 다운로드 및 변환 중: T13_GG_PURPO

## 카드
- cp949
- astype(str) 제거
- on_bad_lines='skip' 제거
- engine='pyarrow' 추가
- sep='|' 추가

In [ ]:
def smart_convert(csv_path):
    """인코딩을 바꿔가며 시도하고 성공 시 CSV 삭제"""
    try:
        if os.path.getsize(csv_path) == 0:
            os.remove(csv_path)
            return False
            
        parquet_path = csv_path.replace('.csv', '.parquet')
        success = False

        # 인코딩 cp949로 지정
        enc = 'cp949'
        try:
            df = pd.read_csv(csv_path, sep='|', encoding=enc, engine='pyarrow') # pyarrow 사용하면 속도 빨라짐
            df.to_parquet(parquet_path, engine='pyarrow', compression='snappy')
            success = True
        except Exception as e:
            print(f"❌ {csv_path} 변환 중 에러 발생: {e}") # 에러 메시지 출력!
        
        if success:
            os.remove(csv_path) # 성공 즉시 CSV 삭제 (용량 확보)
            return True
        return False
    except Exception as e:
        print(f"❌ {csv_path} 변환 중 에러 발생: {e}") # 에러 메시지 출력!
        return False

def download_and_convert_folder(folder_id, local_path):
    if not os.path.exists(local_path):
        os.makedirs(local_path, exist_ok=True)

    query = f"'{folder_id}' in parents and mimeType != 'application/vnd.google-apps.folder' and trashed = False"
    file_list = drive.ListFile({'q': query}).GetList()

    for file_info in file_list:
        file_name = file_info['title']
        file_id = file_info['id']
        parquet_path = os.path.join(local_path, file_name.replace('.csv', '.parquet'))
        csv_save_path = os.path.join(local_path, file_name)

        # 이미 완료된 파일(parquet)이 있으면 무조건 스킵
        if os.path.exists(parquet_path):
            continue

        for attempt in range(3): # 예외가 생길 혹시 모를 경우를 대비해 3번까지 시도
            try:
                print(f"📥 다운로드 및 변환 중: {file_name}")
                file_obj = drive.CreateFile({'id': file_id})
                file_obj.GetContentFile(csv_save_path)
                
                if smart_convert(csv_save_path):
                    print(f"✅ 완료: {file_name}")
                else:
                    print(f"❌ 변환 실패: {file_name}")
                break
            except Exception as e:
                if "quotaExceeded" in str(e).lower():
                    print("구글 할당량 초과! 5분 휴식")
                    time.sleep(300)
                elif "No space left" in str(e):
                    print("하드 용량 부족! 여기서 멈춥니다.")
                    return # 용량 없으면 아예 종료
                else:
                    print(f"⚠️ 에러: {e}")
                    time.sleep(10)

In [ ]:
# --- 메인 실행 루프 ---
base_output = "seongnam_data/카드"

for folder_info in list_of_folders_card:
    name = folder_info['name']
    f_id = folder_info['id']
    target_path = os.path.join(base_output, name)
    
    print(f"\n🚀 {name} 작업 시작...")
    download_and_convert_folder(f_id, target_path)


🚀 10. 매출(대민)(day) 작업 시작...
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2507_성남시.csv
✅ 완료: tbsh_gyeonggi_day_2507_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2506_성남시.csv
✅ 완료: tbsh_gyeonggi_day_2506_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2505_성남시.csv
✅ 완료: tbsh_gyeonggi_day_2505_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2504_성남시.csv
✅ 완료: tbsh_gyeonggi_day_2504_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2503_성남시.csv
✅ 완료: tbsh_gyeonggi_day_2503_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2502_성남시.csv
✅ 완료: tbsh_gyeonggi_day_2502_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2501_성남시.csv
✅ 완료: tbsh_gyeonggi_day_2501_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2412_성남시.csv
✅ 완료: tbsh_gyeonggi_day_2412_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2411_성남시.csv
✅ 완료: tbsh_gyeonggi_day_2411_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2410_성남시.csv
✅ 완료: tbsh_gyeonggi_day_2410_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2409_성남시.csv
✅ 완료: tbsh_gyeonggi_day_2409_성남시.csv
📥 다운로드 및 변환 중: tbsh_gyeonggi_day_2408_성남시.csv
✅ 완료: tbsh_gy

In [ ]:
card = pd.read_parquet('./seongnam_data/카드/10. 매출(대민)(day)/tbsh_gyeonggi_day_2510_성남시.parquet', engine='fastparquet')
card # 해당 카드 폴더 전체 삭제

,ta_ymd,cty_rgn_no,admi_cty_no,card_tpbuz_cd,card_tpbuz_nm_1,card_tpbuz_nm_2,hour,sex,age,day,amt,cnt
0,20251001,41131,41131510,D01,소매/유통,가전제품,3,M,5,3,803213,2
1,20251001,41131,41131510,D01,소매/유통,가전제품,3,M,7,3,68050,2
2,20251001,41131,41131510,D01,소매/유통,가전제품,4,M,6,3,37929,2
3,20251001,41131,41131510,D01,소매/유통,가전제품,4,M,8,3,800982,2
4,20251001,41131,41131510,D01,소매/유통,가전제품,5,M,5,3,44623,2
...,...,...,...,...,...,...,...,...,...,...,...,...
2435212,20251031,41135,41135680,Y04,공공/기업/단체,단체,6,M,5,5,37177,2
2435213,20251031,41135,41135680,Y04,공공/기업/단체,단체,6,M,9,5,6408,2
2435214,20251031,41135,41135680,Y04,공공/기업/단체,단체,7,F,4,5,7264,2
2435215,20251031,41135,41135680,Y04,공공/기업/단체,단체,7,F,6,5,7328,2


In [ ]:
mer_s_2301 = pd.read_parquet('seongnam_data/카드/3. 가맹점 정보(대민)(mer_s)/tbsh_gyeonggi_mer_s_2409_성남시.parquet', engine='fastparquet')
mer_s_2301

In [ ]:
mer_s_2301.info()

In [ ]:
tbsh_2301 = pd.read_parquet('seongnam_data/카드/10. 매출(대민)(day)/tbsh_gyeonggi_day_2301_성남시.parquet')
tbsh_2301.info()

In [ ]:
tbsh_2301_csv = pd.read_csv('seongnam_data/카드/10. 매출(대민)(day)/tbsh_gyeonggi_day_2301_성남시.csv', encoding='cp949')
tbsh_2301_csv

In [ ]:
nps_move_cnt_2405 = pd.read_parquet('seongnam_data/기업/5. 기업 이전 통계(nps_move_cnt)/gg_corp5_nps_move_cnt_202405_성남시.parquet')
nps_move_cnt_2405

In [ ]:
import os
import pandas as pd

def migrate_csv_to_parquet(base_path):
    print(f"🔍 {base_path} 내의 CSV 파일 변환을 시작합니다...")
    success_count = 0

    for root, dirs, files in os.walk(base_path):
        for file in files:
            if not file.endswith('.csv'): continue
            csv_path = os.path.join(root, file)
            parquet_path = csv_path.replace('.csv', '.parquet')

            if os.path.exists(parquet_path): continue
            if os.path.getsize(csv_path) == 0:
                print(f"⚠️ 빈 파일 건너뜀: {file}")
                continue

            try:
                with open(csv_path, 'r', encoding='cp949') as f:
                    df = pd.read_csv(f, sep='|', encoding='cp949')
                if df.empty: continue

                # PyArrow 충돌 시 fastparquet으로 우회
                try:
                    df.to_parquet(parquet_path, engine='pyarrow', compression='snappy')
                except Exception:
                    df.to_parquet(parquet_path, engine='fastparquet', compression='snappy')

                os.remove(csv_path)
                print(f"✅ 변환 완료: {file}")
                success_count += 1
            except Exception as e:
                print(f"❌ 실패 ({file}): {e}")

    print(f"✨ 완료! {success_count}개 변환 성공.")

migrate_csv_to_parquet("seongnam_data/통신")